<a href="https://colab.research.google.com/github/AleksarDo/dj_mln_30-08-25/blob/main/%D1%83%D1%81%D1%82%D0%B0%D0%B2%D0%BA%D0%B0%D0%A0%D0%97%D0%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Выбор уставки по току для МТЗ

In [13]:
# @title
from IPython.display import Markdown, Math, display


Необходимые данные для нахождения тока сработки защиты, $I_{с.з.}$:
- напряжение ступени сети (0,4, 10,5 ...);
- мощность защищаемого трансформатора ТП10/0,4, $S_{тр}$, кВА;
- коэффициент перегруки трансфарматара (по паспортным данным тр-ра) $k_{п.тр.}$;
- коэффициенты: $k_{н}$ - коэф. надёжности (1,1 - 1,2), $k_{сз}$ - коэф. самозапуска двигательной нагрузки (1,1 - 1,3), $k_{в}$ - коэф. возврата защиты (0,95 - 0,97);

In [14]:
# @title

# функция ввода целого числа, все приводится к целому
def inp2int(strii):
    while True:
        inpt_str = input(strii)  # Запрос ввода
        try:
            # Замена запятой на точку, преобразование в float, округление и преобразование в int
            return int(round(float(inpt_str.replace(',', '.')), 0))
        except ValueError:
            print('!! Введите только цифры ')

def inp2flt(strii):
    while True:
        inpt_str = input(strii)  # Запрос ввода
        try:
            # Замена запятой на точку, преобразование в float, округление и преобразование в int
            return round(float(inpt_str.replace(',', '.')), 2)
        except ValueError:
            print('!! ведите только цифры и точку(например, 3.14 или 5) ')



In [15]:
# @title

#Ввод данных:

Ust = inp2flt('напряжение расчетной ступени,кВ = ')
display(Markdown("   $U_{с}$ = "))
print(f'{Ust} кВ')

напряжение расчетной ступени,кВ = 10,5


   $U_{с}$ = 

10.5 кВ


In [16]:
# @title
#Ввод данных:

i3_max = inp2flt('максимальное значение тока 3-хфазного к.з. на шинах 10кВ питающей ПС, кА = ')
display(Markdown("   $I_{к.з.(3)макс}$ = "))
print(f'{i3_max} кA')

максимальное значение тока 3-хфазного к.з. на шинах 10кВ питающей ПС, кА = 4,37


   $I_{к.з.(3)макс}$ = 

4.37 кA


In [17]:
# @title
#Ввод данных:

i3_min = inp2flt('мминимальное значение тока 3-хфазного к.з. на шинах 10кВ питающей ПС, кА = ')
display(Markdown("   $I_{к.з.(3)мин}$ = "))
print(f'{i3_min} кA')

мминимальное значение тока 3-хфазного к.з. на шинах 10кВ питающей ПС, кА = 4,1


   $I_{к.з.(3)мин}$ = 

4.1 кA


In [18]:
# @title
#Ввод данных:

n_uch_sety = inp2int('число участков сети (1 или 2, 3, 4 ... ) - ')
print(f"принимаем - {n_uch_sety} ")

число участков сети (1 или 2, 3, 4 ... ) - 1
принимаем - 1 


In [19]:
# @title
#генератор списка данных для участков рассчитываемой ступени
if n_uch_sety:
    print('net', n_uch_sety)
else:
    print('ect', n_uch_sety)

net 1


In [20]:
# @title

kper = inp2flt('ожидаемый коэффициент перегрузки по справочным данным трансформатора\
 тр-тра = ')
display(Markdown("$k_{пер.тр.}$"))
print(f'принят -  {kper} ')

ожидаемый коэффициент перегрузки по справочным данным трансформатора тр-тра = 1,2


$k_{пер.тр.}$

принят -  1.2 


In [21]:
# @title

Sntr = inp2flt('Номинальная мощность тр-тра = ')
display(Markdown("   $S_{н.тр.}$ = "))
print(f'{Sntr} кВА')

Номинальная мощность тр-тра = 630


   $S_{н.тр.}$ = 

630.0 кВА


Для отстройки уставок защиты определим максимальный рабочий ток линии
$$
I_{макс.раб.}=\frac{S_{н.тр.}}{\sqrt{3}\cdot U_{с}  } \cdot k_{пер.тр}
$$

In [22]:

# @title
#находим максимальный рабочий ток нагрузки
# это Iн тр-ра * kперегрузки для кВА/кВ - будет в А.
Irm = round(kper*Sntr/Ust/3**0.5,2)
display(Markdown("$I_{макс.раб.}$ = "))
print(f'{Irm} А')

$I_{макс.раб.}$ = 

41.57 А


In [23]:
# @title
def calculate_Isz (kn = 1.1, ksz = 1.1, kv = 0.96):
    return round(kn*ksz*Irm/kv, 2)

Для для микропроцессорного блока защиты выключателя значение коэффициента надежности - $k_{н}$ принимаем $\:$ 1,1;

проектируемая линия питает ТП общественного здания без мощных электродвигателей, $k_{сз}$ принимаем 1,1;

для $k_{в}$ выбираем среднее значение - 0,96.

 Определим уставку $\; I_{с.з.}\;$:
 $$
 I_{с.з.}=\frac{k_{н} \cdot k_{сз}}{k_{в}}\cdot I_{макс.раб.}
 $$
далее для вычисленного значения требуется выбрать ток уставки МТЗ...




---

➕
# ***дадаць вывад прадусталяваных каэфіцыентаў і магчымасць увода сваіх!!!***

---



In [24]:
# @title
display(Markdown("   $I_{с.з.}$ = "))
print(f"{calculate_Isz()} A" )
I_sz = input('принимаем значение - ')
print(f'Уставка МТЗ по току принята {I_sz} А.')

   $I_{с.з.}$ = 

52.4 A
принимаем значение - 53
Уставка МТЗ по току принята 53 А.


# Проверка уставки МТЗ


Проверяем коэффициент чувствительности при выбранном значении уставки
$$
k_{ч.з.} = \frac {I_{к.з.min}}{I_{с.з.}} \geq \;1,5
$$
В качестве $\;I_{к.з.min}\;$ принимаем значение тока двухфазного короткого замыкания за тансформатором


- трэба дадаць разлікі тока к.з. 3-фазного, 2-х фазного, 1-но фазного
- ввод сеченій проводніков (кабельных ліній по участкам)
- разлік сапротыўлення лініі
- разлік сапратыўлення трансфарматара
- падлік праверак уставак і г. д.
